# Thống kê corpus

Bảng và biểu đồ được tạo từ JSON khi build. Tỷ lệ dùng thang 0–1; bảng hiển thị thêm phần trăm. Null nghĩa chưa có mẫu số/số đo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from report_utils import load_run
run_name, summary, benchmark, samples = load_run(ROOT)
display(Markdown(f"**Run: {run_name} — label: {benchmark['label']}**"))
if benchmark['label'] == 'synthetic_demo':
    display(Markdown('> **DEMO OFFLINE: các con số không phải kết quả corpus thật hoặc benchmark Colab.**'))
elif benchmark['label'] == 'colab_cpu_sample':
    display(Markdown('> **MẪU COLAB CPU-ONLY: một paper được giao, không phải toàn bộ assignment 25.000 ID.**'))
else:
    display(Markdown('> Lượt gọi API thật; cần đối chiếu môi trường Colab CPU-only, phạm vi input và logs trước nghiệm thu.'))


In [ ]:
display(pd.DataFrame([summary['counts']]))
coverage_path = ROOT / 'summaries/milestone1' / run_name / 'corpus_coverage.json'
if coverage_path.exists():
    import json
    display(Markdown('**Độ bao phủ assignment đầy đủ (khác phạm vi mẫu benchmark):**'))
    display(pd.DataFrame([json.loads(coverage_path.read_text(encoding='utf-8'))]))
rates = pd.DataFrame([{'metric': k, **v} for k,v in summary['rates'].items()])
rates['percent'] = rates['value'] * 100
display(rates)

## Trạng thái theo version

Download và extract dùng cùng tập version đã biết. Discovery lỗi được báo riêng vì chưa biết số version.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax,key,title in zip(axes,['download_statuses','extract_statuses'],['Download outcomes','Extraction outcomes']):
    values = summary[key]
    if values: ax.bar(list(values),list(values.values()))
    ax.set_title(title); ax.set_ylabel('Paper-version count'); ax.tick_params(axis='x',rotation=25)
plt.tight_layout(); plt.show()
display(pd.DataFrame([summary['discovery_statuses']]))

## Kích thước và references

Trước/sau lọc là bytes đã giải nén, tổng mọi version mỗi paper trên cùng tập có đủ cặp đo. Kích thước package nén được báo riêng.

In [ ]:
display(pd.DataFrame([summary['sizes']]))
values = [summary['sizes']['mean_uncompressed_before_per_paper_bytes'], summary['sizes']['mean_filtered_per_paper_bytes']]
if all(v is not None for v in values):
    plt.figure(figsize=(7,4)); plt.bar(['Before filtering','After filtering'],values); plt.ylabel('Mean uncompressed bytes / paired paper'); plt.tight_layout(); plt.show()
packages = [v['package_bytes'] for v in summary['versions'] if v['package_bytes'] is not None]
if packages:
    plt.figure(figsize=(7,4)); plt.hist(packages,bins=min(20,len(packages))); plt.xlabel('Compressed package bytes'); plt.ylabel('Paper-version count'); plt.tight_layout(); plt.show()
display(pd.DataFrame([{'mean_refs_per_retrieved_paper': summary['references']['mean_per_retrieved_paper'], **summary['references']['complete_metadata_rate']}]))
display(pd.DataFrame([summary['reference_statuses']]))

## Nguyên nhân lỗi và dữ liệu thiếu

File/chỉ số không đọc được hiện trong diagnostics. Validator pass được trình bày riêng với source success.

In [ ]:
display(pd.DataFrame([{'cause': k, 'count': v} for k,v in summary['failure_causes'].items()]))
display(pd.DataFrame(summary['diagnostics']))
display(pd.DataFrame(summary['papers']))